# Stage 7 Full Replay Hardmix V2

Full training amount is preserved: `NUM_STEPS=1000`, `MAX_SEQ_LEN=8192`, and `TARGET_REPLAY_ANSWER_TOKENS=2_000_000`. This notebook only changes the data ordering/mixing report and packaging diagnostics. It does not submit the competition.


# Nemotron finetuning pipeline

In [ ]:
# ── Shared config ─────────────────────────────────────────────────────
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

MAX_SEQ_LEN = 8192
NUM_STEPS = 1000
BATCH_SIZE = 32
MICRO_BATCH_SIZE = 4
LEARNING_RATE =  3.5e-4 #2e-4
RESET_WEIGHTS = (
    True  # if True, skip loading pretrained adapter; train from fresh LoRA init
)
IN_PROJ_ONLY = False
MOE_TIE_WEIGHTS = True  # if True, tie one side of MoE expert LoRA across all 128 experts (Tinker-style)
ORIGINAL_PROBLEMS_ONLY = (
    False  # if True, filter examples to only problem_ids listed in train.csv
)
SHUFFLE_DATASET = False

# Stage 7 full training controls. Do not reduce training amount here.
STAGE7_FULL_TRAINING_MODE = "full_replay_hardmix_v2"
STAGE7_KEEP_FULL_STEPS = True
STAGE7_EXPECTED_NUM_STEPS = 1000
STAGE7_EXPECTED_TARGET_REPLAY_ANSWER_TOKENS = 2_000_000

KAGGLE_DATASET = "huikang/nemotron-data"
MINUTES = 60

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "up_proj",
    "down_proj",
    "in_proj",
    "out_proj",
    "lm_head",
]


## Original  Dataset Token Statistics

This script reads the training dataset and checks each example before training. It counts how many valid examples and tokens are available after applying the maximum sequence length limit. It also shows how many full training batches can be made using a batch size of 32.

In [ ]:
import json

MATH_REPLAY_PATH = "/kaggle/input/datasets/mohamedamr992/replay-math/nemotron_math_1gb.jsonl"

row_count = 0

with open(MATH_REPLAY_PATH, "r") as f:
    for _ in f:
        row_count += 1

print("Total rows:", row_count)


In [ ]:
# import os
# import json

# MAX_SEQ_LEN = 8192

# CORPUS_PATH = "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/nemotron-master/training/sft/04-08-16-14/tokens"
# TRAIN_ORDER_PATH = "/kaggle/input/datasets/huikang/huikang-nemotron-repository-snapshot/nemotron-master/training/sft/04-08-16-14/logprobs/index.jsonl"

# ordered_ids = []
# seen = set()

# with open(TRAIN_ORDER_PATH) as f:
#     for line in f:
#         rec = json.loads(line)
#         if rec.get("epoch", 0) != 0:
#             continue
#         pid = rec["problem_id"]
#         if pid in seen:
#             continue
#         seen.add(pid)
#         ordered_ids.append(pid)

# valid_examples = 0
# total_tokens = 0
# total_unmasked = 0

# for sid in ordered_ids:
#     seg_path = os.path.join(CORPUS_PATH, sid, "synthetic.json")
#     if not os.path.isfile(seg_path):
#         continue

#     with open(seg_path) as f:
#         rec = json.load(f)

#     tokens = rec["tokens"]
#     mask = rec["mask"]

#     if not tokens:
#         continue

#     if len(tokens) > MAX_SEQ_LEN:
#         tokens = tokens[:MAX_SEQ_LEN]
#         mask = mask[:MAX_SEQ_LEN]

#     if not any(mask):
#         continue

#     valid_examples += 1
#     total_tokens += len(tokens) - 1
#     total_unmasked += sum(mask[1:])

# print("Ordered problem IDs:", len(ordered_ids))
# print("Valid training examples:", valid_examples)
# print("Total training tokens:", f"{total_tokens:,}")
# print("Total unmasked target tokens:", f"{total_unmasked:,}")
# print("Max full batches at batch size 32:", valid_examples // 32)


In [ ]:
import json

MATH_REPLAY_PATH = "/kaggle/input/datasets/mohamedamr992/replay-math/nemotron_math_1gb.jsonl"

shown = 0

with open(MATH_REPLAY_PATH, "r") as f:
    for line in f:
        row = json.loads(line)

        print("=" * 100)
        print("ROW", shown + 1)
        print("KEYS:", list(row.keys()))

        print("\nPROBLEM:\n")
        print(str(row.get("problem", ""))[:1500])

        print("\nEXPECTED ANSWER:")
        print(row.get("expected_answer"))

        print("\nMESSAGES:\n")
        messages = row.get("messages")
        print(str(messages)[:4000])

        shown += 1
        if shown >= 2:
            break


# Load the model 

In [ ]:
import os
import sys
import subprocess

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_MODAL_WORKER = "MODAL_TASK_ID" in os.environ

if IS_KAGGLE:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-index",
            "--find-links",
            "/kaggle/input/datasets/mayukh18/nemotron-packages/packages",
            "unsloth",
            "trl",
            "peft",
            "transformers",
            "datasets",
            "accelerate",
            "bitsandbytes",
        ],
        check=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        ],
        check=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        ],
        check=True,
    )

print("Environment packages installed.")


In [ ]:
import gc
import torch
import kagglehub

from unsloth import FastLanguageModel
import causal_conv1d
import mamba_ssm
from causal_conv1d import causal_conv1d_fn

MAX_SEQ_LEN = 8192

MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)

print("MODEL_PATH:", MODEL_PATH)

cc = torch.cuda.get_device_capability(0)

print(f"GPU: {torch.cuda.get_device_name(0)}, sm_{cc[0] * 10 + cc[1]}")
print(f"torch={torch.__version__}, cuda={torch.version.cuda}")
print(
    f"mamba_ssm={mamba_ssm.__version__}, "
    f"causal_conv1d={causal_conv1d.__version__}"
)
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

_x = torch.randn(1, 256, 32, device="cuda", dtype=torch.bfloat16)
_w = torch.randn(256, 4, device="cuda", dtype=torch.bfloat16)
causal_conv1d_fn(_x, _w, None, activation="silu")

print("causal_conv1d CUDA kernel: OK")

gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    load_in_8bit=False,
    full_finetuning=False,
    trust_remote_code=True,
    unsloth_force_compile=True,
    attn_implementation="eager",
    dtype=torch.bfloat16,
)

print("Model and tokenizer loaded successfully.")
print("Tokenizer type:", type(tokenizer))


In [ ]:
import json

MATH_REPLAY_PATH = "/kaggle/input/datasets/mohamedamr992/replay-math/nemotron_math_1gb.jsonl"

with open(MATH_REPLAY_PATH, "r") as f:
    row = json.loads(next(f))

messages = row["messages"]

rendered = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False,
)

reasoning = messages[1].get("reasoning_content", "")
final_content = messages[1].get("content", "")

print("Reasoning exists in row:", bool(reasoning))
print("Final answer exists in row:", bool(final_content))
print("Reasoning appears in rendered template:", reasoning[:100] in rendered)
print("Final content appears in rendered template:", final_content[:100] in rendered)

print("\n--- Rendered preview ---\n")
print(rendered[:4000])


## Tonkize the Replay data 

In [ ]:
import json
import os
from tqdm.auto import tqdm

MATH_REPLAY_PATH = "/kaggle/input/datasets/mohamedamr992/replay-math/nemotron_math_1gb.jsonl"
MATH_TOKENIZED_PATH = "/kaggle/working/replay_math_tokenized.jsonl"

MAX_SEQ_LEN = 8192
TARGET_REPLAY_ANSWER_TOKENS = 2_000_000

kept = 0
skipped = 0
truncated = 0
total_tokens = 0
total_answer_tokens = 0

with open(MATH_REPLAY_PATH, "r") as fin, open(MATH_TOKENIZED_PATH, "w") as fout:
    for line in tqdm(fin):
        row = json.loads(line)
        messages = row.get("messages")

        if not messages or len(messages) < 2:
            skipped += 1
            continue

        # Prompt only: user side, ending right before assistant generation begins
        prompt_ids = tokenizer.apply_chat_template(
            messages[:-1],
            tokenize=True,
            add_generation_prompt=True,
        )

        # Full example: user + assistant reasoning + assistant final content
        full_ids = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
        )

        if len(full_ids) <= len(prompt_ids):
            skipped += 1
            continue

        if len(full_ids) > MAX_SEQ_LEN:
            full_ids = full_ids[:MAX_SEQ_LEN]
            truncated += 1

        prompt_len = min(len(prompt_ids), len(full_ids))
        mask = [0] * prompt_len + [1] * (len(full_ids) - prompt_len)

        if len(full_ids) < 2 or not any(mask):
            skipped += 1
            continue

        answer_tokens = sum(mask)

        fout.write(json.dumps({
            "tokens": full_ids,
            "mask": mask
        }) + "\n")

        kept += 1
        total_tokens += len(full_ids)
        total_answer_tokens += answer_tokens

        if total_answer_tokens >= TARGET_REPLAY_ANSWER_TOKENS:
            break

print("Replay examples kept:", kept)
print("Replay examples skipped:", skipped)
print("Truncated examples:", truncated)
print("Total replay tokens:", f"{total_tokens:,}")
print("Trainable replay answer tokens:", f"{total_answer_tokens:,}")
print("Saved to:", MATH_TOKENIZED_PATH)


# Training the model 

In [ ]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_MODAL_WORKER = "MODAL_TASK_ID" in os.environ
IS_MODAL_LAUNCHER = not IS_KAGGLE and not IS_MODAL_WORKER


In [ ]:
# ── Env-specific install (Kaggle only; Modal image has packages pre-installed) ──
if IS_KAGGLE:
    import subprocess

    subprocess.run(
        "pip install -q --no-index --find-links /kaggle/input/datasets/mayukh18/nemotron-packages/packages "
        "unsloth trl peft transformers datasets accelerate bitsandbytes",
        shell=True,
        check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True,
        check=True,
    )
    subprocess.run(
        "pip install -q /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl",
        shell=True,
        check=True,
    )
    for _wd in ["/kaggle/input/datasets/llkh0a/rtx-wheels/wheels"]:
        if os.path.isdir(_wd):
            subprocess.run(
                [
                    "pip",
                    "install",
                    "-q",
                    "--no-index",
                    "--find-links",
                    _wd,
                    "protobuf==6.33.5",
                    "sentencepiece",
                    "safetensors",
                    "huggingface_hub",
                ],
                check=False,
            )
    subprocess.run("rm -rf /kaggle/tmp/*", shell=True, check=True)


In [ ]:
# On Kaggle, trigger training directly after cells load.
# On Modal worker, Modal's runtime calls train_remote() which calls run_training().
import gc
import torch

globals().pop('model', None)
gc.collect()
torch.cuda.empty_cache()

print("Temporary tokenization model cleared from GPU.")
# On Modal launcher, neither fires (main() submits the remote call instead).
if IS_KAGGLE:
    run_training()
